# Regression Algorithms

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [ ]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['Price'] = data.target

print("Dataset: California Housing")
print(f"Shape: {df.shape}")
print(f"\nTarget: Price (Median house value in $100,000s)")
print(f"\nFeatures:")
for i, col in enumerate(data.feature_names):
    print(f"  {i+1}. {col}")

## EDA AND DATA CLEANING

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of Target Variable
plt.figure(figsize=(10, 5))
sns.histplot(df['Price'], kde=True, bins=50, color='steelblue')
plt.title('Distribution of House Prices')
plt.xlabel('Price ($100,000s)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot of selected features vs Price
selected_features = ['MedInc', 'AveRooms', 'HouseAge', 'Price']
sns.pairplot(df[selected_features], diag_kind='kde')
plt.suptitle('Pairplot of Key Features', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot to check for outliers
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, col in enumerate(df.columns[:-1]):
    ax = axes[i // 4, i % 4]
    sns.boxplot(y=df[col], ax=ax, color='lightcoral')
    ax.set_title(col)
plt.suptitle('Boxplots of Features (Outlier Detection)', fontsize=14)
plt.tight_layout()
plt.show()

## FEATURE ENGINEERING & TRAIN-TEST SPLIT

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop('Price', axis=1)
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Helper function to evaluate models
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    r2 = r2_score(y_te, y_pred)
    mae = mean_absolute_error(y_te, y_pred)
    mse = mean_squared_error(y_te, y_pred)
    rmse = np.sqrt(mse)
    print(f"--- {name} ---")
    print(f"  R² Score : {r2:.4f}")
    print(f"  MAE      : {mae:.4f}")
    print(f"  MSE      : {mse:.4f}")
    print(f"  RMSE     : {rmse:.4f}")
    print()
    return {'Model': name, 'R2': r2, 'MAE': mae, 'MSE': mse, 'RMSE': rmse}

In [ ]:
# Store all results for comparison
results = []

## 1. LINEAR REGRESSION

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
res = evaluate_model('Linear Regression', lr, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

## 2. RIDGE REGRESSION (L2 Regularization)

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)
res = evaluate_model('Ridge Regression', ridge, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

## 3. LASSO REGRESSION (L1 Regularization)

In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.01)
res = evaluate_model('Lasso Regression', lasso, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

## 4. ELASTIC NET REGRESSION (L1 + L2)

In [ ]:
from sklearn.linear_model import ElasticNet

elastic = ElasticNet(alpha=0.01, l1_ratio=0.5)
res = evaluate_model('ElasticNet Regression', elastic, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

## 5. POLYNOMIAL REGRESSION

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

lr_poly = LinearRegression()
res = evaluate_model('Polynomial Regression (deg=2)', lr_poly, X_train_poly, X_test_poly, y_train, y_test)
results.append(res)

## 6. SUPPORT VECTOR REGRESSION (SVR)

In [ ]:
from sklearn.svm import SVR

svr = SVR(kernel='rbf', C=1.0, epsilon=0.1)
res = evaluate_model('SVR (RBF Kernel)', svr, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

## 7. DECISION TREE REGRESSOR

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(max_depth=10, random_state=42)
res = evaluate_model('Decision Tree Regressor', dt, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

## 8. RANDOM FOREST REGRESSOR

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42)
res = evaluate_model('Random Forest Regressor', rf, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

## 9. GRADIENT BOOSTING REGRESSOR

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
res = evaluate_model('Gradient Boosting Regressor', gb, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

## 10. ADABOOST REGRESSOR

In [ ]:
from sklearn.ensemble import AdaBoostRegressor

ada = AdaBoostRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
res = evaluate_model('AdaBoost Regressor', ada, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

## 11. XGBOOST REGRESSOR

In [ ]:
!pip install xgboost -q

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, verbosity=0)
res = evaluate_model('XGBoost Regressor', xgb, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

## 12. K-NEAREST NEIGHBORS REGRESSOR

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

knn = KNeighborsRegressor(n_neighbors=5)
res = evaluate_model('KNN Regressor', knn, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

## MODEL COMPARISON

In [ ]:
# Create comparison DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('R2', ascending=False).reset_index(drop=True)
results_df.index = results_df.index + 1  # Start index from 1 (rank)
results_df.index.name = 'Rank'
results_df

In [ ]:
# R² Score Comparison Bar Chart
plt.figure(figsize=(14, 7))
colors = sns.color_palette('viridis', len(results_df))
bars = plt.barh(results_df['Model'], results_df['R2'], color=colors)
plt.xlabel('R² Score')
plt.title('R² Score Comparison of Regression Models')
plt.xlim(0, 1)

# Add value labels on bars
for bar, val in zip(bars, results_df['R2']):
    plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# RMSE Comparison Bar Chart
results_df_rmse = results_df.sort_values('RMSE', ascending=True)

plt.figure(figsize=(14, 7))
colors = sns.color_palette('magma', len(results_df_rmse))
bars = plt.barh(results_df_rmse['Model'], results_df_rmse['RMSE'], color=colors)
plt.xlabel('RMSE')
plt.title('RMSE Comparison of Regression Models (Lower is Better)')

# Add value labels on bars
for bar, val in zip(bars, results_df_rmse['RMSE']):
    plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted for the Best Model
best_model_name = results_df.iloc[0]['Model']
print(f"Best Model: {best_model_name}")
print(f"R² Score: {results_df.iloc[0]['R2']:.4f}")
print(f"RMSE: {results_df.iloc[0]['RMSE']:.4f}")

In [ ]:
# Scatter plot: Actual vs Predicted for the best model
# Re-train best model to get predictions
best_models = {
    'Linear Regression': lr,
    'Ridge Regression': ridge,
    'Lasso Regression': lasso,
    'ElasticNet Regression': elastic,
    'Polynomial Regression (deg=2)': lr_poly,
    'SVR (RBF Kernel)': svr,
    'Decision Tree Regressor': dt,
    'Random Forest Regressor': rf,
    'Gradient Boosting Regressor': gb,
    'AdaBoost Regressor': ada,
    'XGBoost Regressor': xgb,
    'KNN Regressor': knn
}

best = best_models[best_model_name]

if best_model_name == 'Polynomial Regression (deg=2)':
    y_pred_best = best.predict(X_test_poly)
else:
    y_pred_best = best.predict(X_test_scaled)

plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred_best, alpha=0.4, color='teal', edgecolors='k', linewidth=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title(f'Actual vs Predicted — {best_model_name}')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Residual Plot for the Best Model
residuals = y_test - y_pred_best

plt.figure(figsize=(10, 5))
plt.scatter(y_pred_best, residuals, alpha=0.4, color='coral', edgecolors='k', linewidth=0.3)
plt.axhline(y=0, color='black', linestyle='--', linewidth=1)
plt.xlabel('Predicted Price')
plt.ylabel('Residuals')
plt.title(f'Residual Plot — {best_model_name}')
plt.tight_layout()
plt.show()

## HYPERPARAMETER TUNING (Best Model)

In [ ]:
from sklearn.model_selection import GridSearchCV

# GridSearchCV on Gradient Boosting (typically one of the top performers)
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1]
}

gb_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

gb_grid.fit(X_train_scaled, y_train)

print(f"\nBest Parameters: {gb_grid.best_params_}")
print(f"Best CV R² Score: {gb_grid.best_score_:.4f}")

In [ ]:
# Evaluate the tuned model
y_pred_tuned = gb_grid.predict(X_test_scaled)

r2_tuned = r2_score(y_test, y_pred_tuned)
mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))

print(f"--- Tuned Gradient Boosting ---")
print(f"  R² Score : {r2_tuned:.4f}")
print(f"  MAE      : {mae_tuned:.4f}")
print(f"  RMSE     : {rmse_tuned:.4f}")

## CROSS-VALIDATION SCORES

In [ ]:
from sklearn.model_selection import cross_val_score

cv_models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.01),
    'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42),
    'KNN': KNeighborsRegressor(n_neighbors=5)
}

print(f"{'Model':<25} {'Mean R²':>10} {'Std R²':>10}")
print('-' * 47)

for name, model in cv_models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')
    print(f"{name:<25} {scores.mean():>10.4f} {scores.std():>10.4f}")

## FEATURE IMPORTANCE (Random Forest)

In [ ]:
# Feature Importance from Random Forest
importances = rf.feature_importances_
feature_imp = pd.Series(importances, index=data.feature_names).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
feature_imp.plot(kind='barh', color='teal')
plt.title('Feature Importance — Random Forest Regressor')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## SUMMARY

### Regression Algorithms Covered:
1. **Linear Regression** — Simple baseline
2. **Ridge Regression** — L2 regularization to prevent overfitting
3. **Lasso Regression** — L1 regularization for feature selection
4. **ElasticNet** — Combination of L1 + L2 regularization
5. **Polynomial Regression** — Captures non-linear relationships
6. **Support Vector Regression (SVR)** — Kernel-based regression
7. **Decision Tree Regressor** — Tree-based non-linear model
8. **Random Forest Regressor** — Ensemble of decision trees (bagging)
9. **Gradient Boosting Regressor** — Sequential boosting ensemble
10. **AdaBoost Regressor** — Adaptive boosting
11. **XGBoost Regressor** — Extreme Gradient Boosting
12. **KNN Regressor** — Instance-based learning

### Key Takeaways:
- Ensemble methods (Random Forest, Gradient Boosting, XGBoost) generally outperform simple linear models
- Feature scaling is crucial for distance-based models (SVR, KNN) and regularized models (Ridge, Lasso)
- MedInc (Median Income) is the most important feature for predicting house prices
- Hyperparameter tuning can further improve model performance